# Modeling — comparing four models and picking a champion

Part of the **Personality Profile Predictor** project: predicting a person's Big Five
personality profile from 19 questionnaire statements plus age, gender and writing hand.

This notebook picks the model that goes into the Streamlit app:

1. load the cleaned data from the EDA notebook,
2. build the **preprocessing pipeline** that every model shares,
3. give **each of four models its own pipeline**,
4. score them all with **5-fold cross-validation** on **macro F1**,
5. **tune** the two best ones — with grid search *and* with Hyperopt,
6. pick a **champion**, log every run, and save the pipeline for the app.

The notebook runs from top to bottom.

## 0. Setup

All the imports and the settings we reuse later, in one place.

In [ ]:
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_predict, cross_val_score, cross_validate)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")          # hide sklearn's convergence warnings
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42       # same number everywhere, so results are reproducible
N_SPLITS = 5            # 5-fold cross-validation
SCORING = "f1_macro"    # our comparison metric — see section 3 for why
MAX_EVALS = 15          # how many settings Hyperopt tries per model

# StratifiedKFold keeps the class proportions the same in every fold.
# That matters here because our classes are imbalanced.
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print("Setup ready.")

## 1. Load the cleaned data

The EDA notebook wrote `data/data_clean.csv` after removing duplicates, answers outside the
1–5 scale, rows with missing `gender`/`hand`, and impossible ages.

In [ ]:
df = pd.read_csv("data/data_clean.csv")

X = df.drop(columns="target")   # the features: 19 answers + age + gender + hand
y = df["target"]                # what we want to predict, kept as text

item_cols = [c for c in df.columns if c not in ["age", "gender", "hand", "target"]]

print("Rows:", len(df))
print("Features:", X.shape[1], "| questionnaire items:", len(item_cols))
df.head()

### The classes are imbalanced

This single fact decides which metric we use for the rest of the notebook.

In [ ]:
counts = y.value_counts()

target_overview = pd.DataFrame({
    "count": counts,
    "share_%": (counts / len(y) * 100).round(1),
})
target_overview.index.name = "personality type"
display(target_overview)

plt.figure(figsize=(7, 3.5))
sns.countplot(x=y, order=counts.index, color="#2F4E7E")
plt.title("How many people of each personality type?")
plt.xlabel("")
plt.show()

## 2. The preprocessing pipeline

Models need numbers, with no gaps and on a comparable scale. Our columns need different
treatment:

- **numbers** (the 19 items + `age`): fill any gaps with the median, then scale
- **categories** (`gender`, `hand`): fill any gaps with the most frequent value, then one-hot encode

A `ColumnTransformer` consisting of the individual pipelines (numerical and catogorical transformers) building the mainpreprocessing pipeline describes this once. Every model below reuses it, so each model sees exactly the same prepared data and the comparison is fair.

The target stays as **text** (`"Resilient"`), so predictions come out readable in the app.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        # make_column_selector picks every numeric column automatically
        ('num', numeric_transformer, make_column_selector(dtype_include='number')),
        ('cat', categorical_transformer, ['gender', 'hand'])
    ]
)

preprocessor

### What the preprocessor actually does

`gender` and `hand` turn into several 0/1 columns, and no gaps are left behind.

In [ ]:
X_prepared = preprocessor.fit_transform(X)

print("Before:", X.shape, " After:", X_prepared.shape)
print("Missing values after preprocessing:", int(np.isnan(X_prepared).sum()))

# The names of the columns that come out of the preprocessor
feature_names = preprocessor.get_feature_names_out()
print("Last few columns:", list(feature_names[-6:]))

## 3. The metric: macro F1, not accuracy

Moderate alone is about 43% of the data. A "model" that always predicts Moderate therefore
gets 43% accuracy without learning anything — so accuracy flatters a model that ignores the
small classes.

**Macro F1** averages the F1 score of every class *equally*, so Undercontroller (12% of the
data) counts as much as Moderate. That is our comparison number from here on.

The baseline below makes the gap concrete. It is scored with the same cross-validation as
every real model.

In [ ]:
# A "model" that always predicts the most common class
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent'))
])

baseline_f1 = cross_val_score(baseline_pipeline, X, y, cv=cv, scoring="f1_macro").mean()
baseline_acc = cross_val_score(baseline_pipeline, X, y, cv=cv, scoring="accuracy").mean()

print(f"Baseline (always 'Moderate'):  macro F1 = {baseline_f1:.3f}   accuracy = {baseline_acc:.3f}")
print()
print("So our models have to beat 0.15 macro F1 — not 0.43 accuracy.")

## 4. Four models, four pipelines

Each model gets **its own pipeline**: the shared preprocessor in front, the classifier at the
end. They come from four different families, so the comparison tells us something about the
problem and not just about one algorithm:

| Pipeline | Model | Family | Why |
|---|---|---|---|
| `pipe_logreg` | LogisticRegression | linear | Simple, fast, explainable — the one to beat |
| `pipe_knn` | KNeighborsClassifier | distance-based | Makes no assumption about the shape of the boundary |
| `pipe_rf` | RandomForestClassifier | bagged trees | Handles combinations of answers, resists overfitting |
| `pipe_hgb` | HistGradientBoostingClassifier | boosted trees | Usually the strongest on tabular data of this size |

In [ ]:
pipe_logreg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

pipe_knn = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier())
])

pipe_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

pipe_hgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(random_state=RANDOM_STATE))
])

# A dictionary so we can loop over all four below
pipelines = {
    "LogisticRegression": pipe_logreg,
    "KNN": pipe_knn,
    "RandomForest": pipe_rf,
    "HistGradientBoosting": pipe_hgb,
}

pipe_hgb

## 5. Cross-validation: scoring all four fairly

A single train/test split can be lucky or unlucky. **5-fold cross-validation** trains five
times on different splits and averages the scores, which is a much steadier estimate.

`cross_validate` gives us the macro F1 score *and* accuracy in one go. We also keep the standard deviation across the folds (how stable the score is).

Every result in this notebook goes into the list `runs`, so we can build one leaderboard at
the end.

In [ ]:
runs = []   # every scored model lands here, tuned or not

for name, pipe in pipelines.items():
    print(f"Cross-validating {name} ...")

    scores = cross_validate(pipe, X, y, cv=cv, scoring=["f1_macro", "accuracy"])

    runs.append({
        "model": name,
        "tuning": "none",                              # not tuned yet
        "f1_macro": scores["test_f1_macro"].mean(),    # average over the 5 folds
        "f1_std": scores["test_f1_macro"].std(),       # how much the folds disagree
        "accuracy": scores["test_accuracy"].mean(),
        "params": {},
        "pipeline": pipe,
    })

# Turn the list into a table, best score first
cv_results = pd.DataFrame(runs).sort_values("f1_macro", ascending=False).reset_index(drop=True)

cv_results[["model", "f1_macro", "f1_std", "accuracy"]].round(4)

### Same table as a chart, with the baseline marked

In [ ]:
plot_data = cv_results.sort_values("f1_macro")   # smallest first, so the best bar is on top

plt.figure(figsize=(9, 4))
plt.barh(plot_data["model"], plot_data["f1_macro"], xerr=plot_data["f1_std"],
         color="#2F4E7E", height=0.6)
plt.axvline(baseline_f1, color="#9A4A16", linestyle="--")
plt.text(baseline_f1 + 0.01, -0.45, "baseline(always 'Moderate')", color="#9A4A16", fontsize=9)
plt.xlabel("macro F1 (5-fold cross-validation)")
plt.xlim(0, 1)
plt.title("Model comparison — before tuning")
plt.tight_layout()
plt.show()

### Where the errors are

The best model's overall score hides big differences between the classes.

`cross_val_predict` predicts every row using a model that did **not** see that row during
training. So we get one prediction per person, all of them honest, and we can read a
classification report without needing a separate test split.

In [ ]:
best_name = cv_results.loc[0, "model"]          # the leader after section 5
best_pipeline = pipelines[best_name]

y_pred_cv = cross_val_predict(best_pipeline, X, y, cv=cv)

print(f"Out-of-fold predictions from {best_name}\n")
print(classification_report(y, y_pred_cv, digits=3))

### Confusion matrix: each row shows what share of that true class went where


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
ConfusionMatrixDisplay.from_predictions(
    y, y_pred_cv,
    labels=sorted(y.unique()),
    normalize="true",   # show shares instead of counts
    cmap="Blues",
    values_format=".2f",
    colorbar=False,
    xticks_rotation=45,
    ax=ax,
)
ax.set_title(f"{best_name} — share of each true class")
plt.tight_layout()
plt.show()

**Look at the Undercontroller row.** Its recall is far below the others, and that one class is what holds the macro F1 down.

The reason is in the data, not the model. Undercontroller is defined by low Conscientiousness *and* low Agreeableness, but only two of the 19 statements (`C4` and `A4`) measure those two traits. Most of the signal needed to recognize it was lost when the dataset was reduced from 50 statements to 19.

Tuning cannot invent that signal — but `class_weight="balanced"` tells a model to pay more attention to the rare classes, so we put it in both searches below and let the tuning decide whether it helps.

## 6. Hyperparameter tuning

A hyperparameter is a setting we choose before training (how deep a tree may grow, how strong
the regularisation is). Tuning means searching for the combination that scores best.

We tune only the **two best models**. Tuning the other two would cost minutes to close a gap
they cannot close — and explaining why we stopped is part of the answer.

Two methods, both scored with the same cross-validation and the same macro F1:

| Method | How it searches | Good for |
|---|---|---|
| **GridSearchCV** | Tries every combination in a list we write | Few settings, values we can name in advance |
| **Hyperopt** | Learns from the trials it already ran and tries promising settings next | Many settings, or continuous ranges |

In [ ]:
# The two models with the best macro F1 from section 5
models_to_tune = cv_results.head(2)["model"].tolist()

print("Tuning:", models_to_tune)
print("Not tuning:", [m for m in pipelines if m not in models_to_tune],
      "- too far behind to catch up")

### 6.1 Grid search

`GridSearchCV` scores **every** combination in the grid, so it is exhaustive — but the cost
multiplies. Two settings with 2 and 4 values is already 8 combinations × 5 folds = 40 fits.

Note the `classifier__` prefix in the parameter names. It means *"the `C` of the pipeline step
called `classifier`"*. That is how we reach inside the pipeline. And because the whole
pipeline goes into the search, the preprocessor is refit inside every fold — so nothing leaks
from the validation fold into the scaling.

In [ ]:
param_grids = {
    "LogisticRegression": {
        'classifier__C': [0.1, 1, 10, 100], #c stands for regularization strength
        'classifier__class_weight': [None, 'balanced'], #class_weight is used to handle imbalanced datasets by assigning weights to classes
    },
    "KNN": {
        'classifier__n_neighbors': [5, 15, 30, 50],
        'classifier__weights': ['uniform', 'distance'], #weights parameter determines how the influence of each neighbor is weighted when making predictions. 'uniform' means all neighbors are weighted equally, while 'distance' means closer neighbors have more influence.
    },
    "RandomForest": {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [10, 20, None],
        'classifier__class_weight': [None, 'balanced'],
    },
    "HistGradientBoosting": {
        'classifier__learning_rate': [0.05, 0.1], #learning_rate controls how much the model adjusts in response to each new tree. Lower values can improve accuracy but require more trees.
        'classifier__max_leaf_nodes': [31, 63], #limiting the number of leaf nodes can help prevent overfitting and improve generalization.
        'classifier__class_weight': [None, 'balanced'],
    },
}

for name in models_to_tune:
    print(f"--- Grid search: {name} ---")
    start = time.time()

    grid_search = GridSearchCV(
        estimator=pipelines[name],      # the whole pipeline, not just the classifier
        param_grid=param_grids[name],
        cv=cv,
        scoring=SCORING,
        n_jobs=-1,                      # use all CPU cores
    )
    grid_search.fit(X, y)

    seconds = time.time() - start
    before = cv_results.loc[cv_results["model"] == name, "f1_macro"].iloc[0]

    runs.append({
        "model": name,
        "tuning": "grid search",
        "f1_macro": grid_search.best_score_,
        "f1_std": np.nan,
        "accuracy": np.nan,
        "params": grid_search.best_params_,
        "pipeline": grid_search.best_estimator_,   # already set up with the best settings
    })

    print(f"  combinations tried = {len(grid_search.cv_results_['params'])}")
    print(f"  before tuning      = {before:.4f}")
    print(f"  after tuning       = {grid_search.best_score_:.4f}  ({grid_search.best_score_ - before:+.4f})")
    print(f"  best settings      = {grid_search.best_params_}")
    print(f"  took               = {seconds:.0f}s\n")

### 6.2 Hyperopt

Now the same two models with **continuous** ranges: `learning_rate` anywhere between 0.02 and
0.3, not just the two values the grid could afford. Hyperopt tries a setting, sees the score,
and uses that to decide what to try next — so it covers a much wider space in the same number
of fits.

Three things to know about the syntax below:

- `hp.loguniform` samples across orders of magnitude (0.01, 0.1, 1, 10 are equally likely)
- `scope.int(...)` is needed because `hp.quniform` returns a float, and `n_estimators` must be a whole number
- Hyperopt **minimises**, so the objective returns `-score`

In [ ]:
from hyperopt import STATUS_OK, Trials, fmin, hp, space_eval, tpe
from hyperopt.pyll.base import scope
from sklearn.base import clone

search_spaces = {
    "LogisticRegression": {
        "C": hp.loguniform("C", np.log(0.01), np.log(100)),
        "class_weight": hp.choice("class_weight", [None, "balanced"]),
    },
    "KNN": {
        "n_neighbors": scope.int(hp.quniform("n_neighbors", 5, 60, 5)),
        "weights": hp.choice("weights", ["uniform", "distance"]),
    },
    "RandomForest": {
        "n_estimators": scope.int(hp.quniform("n_estimators", 100, 400, 50)),
        "max_depth": scope.int(hp.quniform("max_depth", 5, 25, 1)),
        "class_weight": hp.choice("class_weight", [None, "balanced"]),
    },
    "HistGradientBoosting": {
        "learning_rate": hp.loguniform("learning_rate", np.log(0.02), np.log(0.3)),
        "max_leaf_nodes": scope.int(hp.quniform("max_leaf_nodes", 15, 127, 16)),
        "min_samples_leaf": scope.int(hp.quniform("min_samples_leaf", 10, 80, 10)),
        "class_weight": hp.choice("class_weight", [None, "balanced"]),
    },
}

print("Search spaces ready.")

In [ ]:
for name in models_to_tune:
    print(f"--- Hyperopt: {name} ---")
    start = time.time()

    pipe = pipelines[name]
    space = search_spaces[name]

    # Hyperopt calls this with one set of settings and wants a score back
    def objective(params):
        candidate = clone(pipe)                 # a fresh copy, so nothing carries over
        # set_params needs the "classifier__" prefix, same as in the grid above
        candidate.set_params(**{"classifier__" + key: value for key, value in params.items()})
        score = cross_val_score(candidate, X, y, cv=cv, scoring=SCORING).mean()
        return {"loss": -score, "status": STATUS_OK}    # minus, because Hyperopt minimises

    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,                       # the smart search strategy
        max_evals=MAX_EVALS,
        trials=trials,
        rstate=np.random.default_rng(RANDOM_STATE),
    )

    # fmin returns indexes for hp.choice, so space_eval turns them back into real values
    best_params = space_eval(space, best)
    best_score = -trials.best_trial["result"]["loss"]    # flip the sign back

    # Build the pipeline with those settings
    tuned_pipeline = clone(pipe)
    tuned_pipeline.set_params(**{"classifier__" + k: v for k, v in best_params.items()})

    seconds = time.time() - start
    before = cv_results.loc[cv_results["model"] == name, "f1_macro"].iloc[0]

    runs.append({
        "model": name,
        "tuning": "hyperopt",
        "f1_macro": best_score,
        "f1_std": np.nan,
        "accuracy": np.nan,
        "params": best_params,
        "pipeline": tuned_pipeline,
    })

    print(f"  trials        = {MAX_EVALS}")
    print(f"  before tuning = {before:.4f}")
    print(f"  after tuning  = {best_score:.4f}  ({best_score - before:+.4f})")
    print(f"  best settings = {best_params}")
    print(f"  took          = {seconds:.0f}s\n")

## 7. The leaderboard and the champion

Every run from this notebook in one table — four untuned models, plus two tuning methods on
the two best ones.

The **champion is simply the row with the highest macro F1**, the metric we committed to in
section 3.

In [ ]:
# All runs, best score first
all_runs = pd.DataFrame(runs).sort_values("f1_macro", ascending=False).reset_index(drop=True)

# The readable table (the pipeline objects stay out of it)
leaderboard = all_runs[["model", "tuning", "f1_macro", "accuracy"]].round(4)
display(leaderboard)

# The winning row — it carries its own pipeline, so we can use it straight away
champion = all_runs.loc[0]

CHAMPION_NAME = champion["model"]
CHAMPION_TUNING = champion["tuning"]
CHAMPION_SCORE = champion["f1_macro"]
champion_pipeline = champion["pipeline"]
champion_params = champion["params"]

print(f"CHAMPION: {CHAMPION_NAME}  (tuning: {CHAMPION_TUNING})")
print(f"Metric:   macro F1, {N_SPLITS}-fold stratified cross-validation")
print(f"Score:    {CHAMPION_SCORE:.4f}   (baseline was {baseline_f1:.4f})")
if champion_params:
    print(f"Settings: {champion_params}")

### How the champion behaves per class

The same out-of-fold check as before, now on the model we are actually shipping.

In [ ]:
y_pred_champion = cross_val_predict(champion_pipeline, X, y, cv=cv)
print(f"{CHAMPION_NAME} ({CHAMPION_TUNING}) — out-of-fold classification report\n")
print(classification_report(y, y_pred_champion, digits=3))

fig, ax = plt.subplots(figsize=(6, 5.5))
ConfusionMatrixDisplay.from_predictions(
    y, y_pred_champion,
    labels=sorted(y.unique()),
    normalize="true",
    cmap="Greens",
    values_format=".2f",
    colorbar=False,
    xticks_rotation=45,
    ax=ax,
)
ax.set_title(f"Champion: {CHAMPION_NAME} ({CHAMPION_TUNING})")
plt.tight_layout()
plt.show()

## 8. Log every run

Two records, so no result gets lost:

- `results/model_comparison.csv` — the leaderboard as a file, which the README quotes
- **MLflow** — one run per model with its settings, its score, and the champion's pipeline

MLflow writes to `mlflow.db` in the project folder. Open the UI **from that same folder**:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

In [ ]:
Path("results").mkdir(exist_ok=True)
leaderboard.to_csv("results/model_comparison.csv", index=False)

print("Saved results/model_comparison.csv")
leaderboard

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("personality-profile-predictor")

for _, run in all_runs.iterrows():
    with mlflow.start_run(run_name=f"{run['model']} - {run['tuning']}"):

        # What was run
        mlflow.log_param("model", run["model"])
        mlflow.log_param("tuning", run["tuning"])
        mlflow.log_param("cv_folds", N_SPLITS)
        mlflow.log_param("metric", SCORING)

        # The settings it used (drop the "classifier__" prefix so the UI reads nicely)
        for key, value in run["params"].items():
            mlflow.log_param(key.replace("classifier__", ""), value)

        # How it scored
        mlflow.log_metric("f1_macro_cv", float(run["f1_macro"]))
        if not pd.isna(run["accuracy"]):
            mlflow.log_metric("accuracy_cv", float(run["accuracy"]))

        # Mark the winner, and store its pipeline as an artifact
        is_champion = (run["model"] == CHAMPION_NAME) and (run["tuning"] == CHAMPION_TUNING)
        mlflow.set_tag("champion", str(is_champion))

        if is_champion:
            mlflow.sklearn.log_model(
                champion_pipeline.fit(X, y),
                artifact_path="model",
                serialization_format="cloudpickle",
            )

print(f"Logged {len(all_runs)} runs to mlflow.db")

## 9. Fit the champion on all the data and save it

Cross-validation told us *which* pipeline to use. Now we refit it on the **whole** dataset —
more training data, same settings — and save it for `app.py`.

We save the **whole pipeline**, not just the classifier. That way the app applies exactly the
same imputing, scaling and encoding that the model was trained with.

In [ ]:
Path("models").mkdir(exist_ok=True)

final_model = champion_pipeline.fit(X, y)
joblib.dump(final_model, "models/personality_pipeline.joblib")

print(f"Saved models/personality_pipeline.joblib  ({CHAMPION_NAME}, {CHAMPION_TUNING})")
print("Classes:", list(final_model.classes_))

# Quick check that it works: predict the first person in the dataset
example = X.head(1)
print("\nExample prediction:", final_model.predict(example)[0])

## 10. For the README

This prints the results section to paste into `README.md`, so the champion and the metric that
chose it are recorded outside the notebook too.

In [ ]:
lines = []
lines.append("## Results")
lines.append("")
lines.append(f"**Champion model: {CHAMPION_NAME} (tuned with {CHAMPION_TUNING})**")
lines.append("")
lines.append(
    f"Selected by **macro F1** under {N_SPLITS}-fold stratified cross-validation — the metric "
    "that treats every personality type equally, which matters because the classes are "
    "imbalanced (Moderate 43%, Undercontroller 12%)."
)
lines.append("")
lines.append(f"- Champion macro F1: **{CHAMPION_SCORE:.3f}**")
lines.append(f"- Baseline (always predicts Moderate): {baseline_f1:.3f} macro F1")
lines.append("")
lines.append("| Model | Tuning | Macro F1 |")
lines.append("|---|---|---|")

for _, run in all_runs.iterrows():
    lines.append(f"| {run['model']} | {run['tuning']} | {run['f1_macro']:.3f} |")

if champion_params:
    lines.append("")
    lines.append(f"Champion settings: `{champion_params}`")

lines.append("")
lines.append(
    "Undercontroller is the hardest class for every model: it is defined by low "
    "Conscientiousness and low Agreeableness, but only two of the 19 statements measure "
    "those traits, so most of the signal needed to identify it is not in the data."
)

readme_section = "\n".join(lines)
Path("results/README_results.md").write_text(readme_section, encoding="utf-8")

print(readme_section)

## Recap

- Four models, each in **its own pipeline**, all sharing one preprocessor.
- Scored with **5-fold cross-validation** on **macro F1**, against a baseline that shows what
  43% accuracy is really worth.
- The two best were tuned twice — once with **GridSearchCV**, once with **Hyperopt** — on the
  same folds and the same metric, so the two methods can be compared directly.
- The **champion** was chosen by that metric, logged to MLflow, and saved to
  `models/personality_pipeline.joblib`.

Next: `streamlit run app.py` loads that file and serves predictions.